# Session 2 — Chat Applications
**Date:** Sun 21 Jun 2026 | **Duration:** 4 hrs (9:30 am – 1:30 pm IST)  
**Course:** Python Applications with OpenAI — MSME Technology Development Center, Chennai

---

## Learning Objectives
By the end of this session you can:
- Explain the three message roles and how they shape model behaviour
- Maintain multi-turn conversation history using a running `messages` list
- Tune key Chat Completions parameters (`temperature`, `max_tokens`, `stop`)
- Build a working CLI chatbot with persistent conversation memory
- Describe the structure of a Streamlit chat UI and run the app skeleton

## Agenda
| Time | Topic |
|------|-------|
| 00:00 | Recap of Session 1 + setup check |
| 00:20 | Message roles deep dive |
| 01:00 | Multi-turn conversation state |
| 01:30 | **BREAK** |
| 01:40 | Key parameters (`temperature`, `max_tokens`, `stop`) |
| 02:20 | Building a CLI chatbot |
| 02:50 | **BREAK** |
| 03:00 | Minimal Streamlit chat UI |
| 03:40 | Exercises & recap |

In [1]:
# ── Session 2 Setup (run this cell first) ────────────────────────────────────
import os
import json
import sys
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

_root = Path(os.getcwd())
for _ in range(5):
    if (_root / "utils").is_dir():
        break
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from utils.config import MODELS, get_client, estimate_cost
from utils.helpers import pretty_print, retry, mock_chat_response

MOCK  = os.getenv("MOCK", "0") == "1"
MODEL = MODELS["chat"]   # gpt-5.4-mini — swap to MODELS["frontier"] for gpt-5.5

if MOCK:
    print(f"MOCK mode ON — no real API calls. Model constant: {MODEL}")
else:
    try:
        client = get_client()
        client.models.list()
        print(f"Client is alive.  Default model: {MODEL}")
    except Exception as e:
        print(f"Setup issue: {e}\nCheck OPENAI_API_KEY in .env")

Client is alive.  Default model: gpt-5.4-mini


---
## 1. Message Roles Deep Dive

Every Chat Completions request is a list of **messages**. Each message has exactly two fields — `role` and `content`.

| Role | Who provides it | Purpose |
|------|-----------------|---------|
| `system` | Developer | Sets the model's persona, tone, and constraints. Sent once at the start of the conversation. |
| `user` | The human | The actual question or instruction. |
| `assistant` | The model (you re-send it) | Previous replies from the model. You must append these yourself to maintain context — the API has **no memory** between calls. |

> **Key insight:** The API is completely stateless. Every call is independent. The only "memory" the model has is whatever you include in the `messages` list you send.

In [9]:
# ── 1a. Use all three roles to seed conversation history ────────────────────
# GOAL: Inject a prior exchange using an "assistant" message so the model
#       can continue a conversation it never actually had
# STEPS:
#   1. messages = [
#        {"role": "system",    "content": "You are a financial advisor for small business owners in India."},
#        {"role": "user",      "content": "What is a GST return?"},
#        {"role": "assistant", "content": "A GST return is a document filed by registered taxpayers..."},
#        {"role": "user",      "content": "How often do I need to file it?"},
#      ]
#   2. Call the API (or mock)
#   3. print(response.choices[0].message.content)
#   4. Observe: the model knows what a GST return is because we injected that history
# --- live-code below ---

messages = [
       {"role": "system",    "content": "You are a financial advisor for small business owners in India."},
       {"role": "user",      "content": "What is a GST return?"},
       {"role": "assistant", "content": "A GST return is a document filed by registered taxpayers..."},
       {"role": "user",      "content": "How often do I need to file it?"},
     ]


response = client.chat.completions.create(messages=messages,model=MODEL)




print(response.choices[0].message.content)





It depends on your GST registration type and turnover, but in India the common filing frequencies are:

- **Monthly**: Most regular taxpayers file **GSTR-1** and **GSTR-3B** every month.
- **Quarterly**: Small taxpayers under the **QRMP scheme** can file **GSTR-1** and **GSTR-3B** quarterly.
- **Annual**: Most registered businesses also file **GSTR-9** yearly, if applicable.

Other returns may apply in special cases, such as:
- **GSTR-4** for composition scheme taxpayers: **once a year**
- **GSTR-5 / 5A / 6 / 7 / 8**: for specific categories like non-resident taxpayers, input service distributors, TDS/TCS, e-commerce, etc.

If you want, I can tell you **which GST returns your business must file** based on your business type.


In [3]:
messages.append({'role':"assistant",'content':response.choices[0].message.content})


messages.append({'role':'user','content':'How often do I need to file it?'})

In [6]:
print(messages)

[{'role': 'system', 'content': 'You are a financial advisor for small business owners in India.'}, {'role': 'user', 'content': 'What is a GST return?'}, {'role': 'assistant', 'content': 'A **GST return** is a statement that a registered business files with the government to report its **sales, purchases, output tax collected, and input tax paid** under GST.\n\nIn simple terms, it tells the tax department:\n- how much you sold,\n- how much GST you collected from customers,\n- how much GST you paid on business purchases,\n- and how much tax you need to pay or can claim as credit.\n\n### Why it matters\nGST returns are used to:\n- calculate your **GST liability**\n- claim **Input Tax Credit (ITC)**\n- keep your business compliant with GST laws\n\n### Common GST returns in India\nDepending on your registration and business type, you may need to file:\n- **GSTR-1** – details of outward sales/invoices\n- **GSTR-3B** – summary return with tax payment\n- **CMP-08** – for composition taxpayers\

In [ ]:
# response = 2

In [15]:
# ── 1b. System prompt controls persona — same question, different identities ─
# GOAL: See how dramatically the system message changes the response
# STEPS:
#   1. user_question = "Explain machine learning in one sentence."
#   2. personas = [
#        "You are a professor of computer science writing for a research journal.",
#        "You are explaining to a 10-year-old child.",
#        "You are a sceptical journalist writing for a general newspaper.",
#      ]
#   3. For each persona:
#        messages = [{"role": "system", "content": persona},
#                    {"role": "user",   "content": user_question}]
#        Call API → print persona label + reply
# --- live-code below ---

from werkzeug.wrappers.response import Response


user_question = 'Explain machine learning in one sentence'

personas = [
       "You are a professor of computer science writing for a research journal.",
       "You are explaining to a 10-year-old child.",
       "You are a sceptical journalist writing for a general newspaper.",
     ]

persona = personas[2]


messages = [
    {"role": "system", "content": persona},
    {"role": "user",   "content": user_question}
]

print(messages)


response = client.chat.completions.create(messages=messages,model=MODEL)


pretty_print(response)


[{'role': 'system', 'content': 'You are a sceptical journalist writing for a general newspaper.'}, {'role': 'user', 'content': 'Explain machine learning in one sentence'}]
────────────────────────────────────────────────────────────
Machine learning is a way of teaching computers to spot patterns in data and make predictions or decisions without being explicitly programmed for every case.
────────────────────────────────────────────────────────────
  Tokens  prompt=28  completion=29  total=57


---
## 2. Multi-Turn Conversation State

Because the API is **stateless**, you maintain context by appending every user message *and* every assistant reply to a shared `messages` list, then re-sending the whole list on the next call.

```python
messages = [{"role": "system", "content": SYSTEM}]

# ── Turn 1 ──
messages.append({"role": "user", "content": user_input_1})
response = client.chat.completions.create(model=MODEL, messages=messages)
messages.append({"role": "assistant", "content": response.choices[0].message.content})

# ── Turn 2 — messages now has 3 entries (system + user1 + assistant1) ──
messages.append({"role": "user", "content": user_input_2})
response = client.chat.completions.create(model=MODEL, messages=messages)
messages.append({"role": "assistant", "content": response.choices[0].message.content})
```

### Cost grows with history
Each turn re-sends all previous turns as prompt tokens. By turn 10 you could be sending 2 000+ prompt tokens per call. Truncation strategies (covered in Session 3):
- Keep only the last N turns
- Summarise older turns into one message
- Use the Responses API's server-side conversation state

In [20]:
# ── 2a. Build a two-turn conversation with real context carry-over ───────────
# GOAL: Prove the model uses history — it should remember what was said in Turn 1
# STEPS:
#   1. messages = [{"role": "system", "content": "You are a helpful assistant."}]
#   2. Turn 1 — user: "My name is Priya and I work in textile manufacturing in Coimbatore."
#      Call API → get reply → append {"role": "assistant", ...} to messages
#      Print reply
#   3. Turn 2 — user: "What's my name and which city am I based in?"
#      Call API → get reply → append
#      Print reply  ← model should correctly recall "Priya" and "Coimbatore"
#   4. print(f"History length: {len(messages)} messages")
# --- live-code below ---

messages = [
    {"role":"system","content":"You are a helpful assistant."},
    {"role":"user","content":"My name is Priya and I work in textile manufacturing in Coimbatore."}
]


response = client.chat.completions.create(messages=messages,model=MODEL)


pretty_print(response)


────────────────────────────────────────────────────────────
Nice to meet you, Priya — Coimbatore is a major textile hub, so that sounds like interesting work.

If you'd like, I can help with things like:
- textile manufacturing terminology
- production/process explanations
- quality control
- machinery basics
- business emails or reports related to textiles

What would you like to talk about?
────────────────────────────────────────────────────────────
  Tokens  prompt=32  completion=73  total=105


In [21]:
print(messages)

[{'role': 'system', 'content': 'You are a helpful assistant.'}, {'role': 'user', 'content': 'My name is Priya and I work in textile manufacturing in Coimbatore.'}]


In [ ]:
messages.append({"role":"assistant","content":response.choices[0].message.content})

messages.append({'role':"user","content":"What's my name and which city am I based in?"})

response = client.chat.completions.create(messages=messages,model=MODEL)

pretty_print(response)

────────────────────────────────────────────────────────────
Your name is Priya, and you’re based in Coimbatore.
────────────────────────────────────────────────────────────
  Tokens  prompt=123  completion=18  total=141


In [ ]:
# ── 2b. Watch prompt tokens grow with each additional turn ───────────────────
# GOAL: Make the cost of multi-turn conversations visible
# STEPS:
#   1. Start fresh: messages = [{"role": "system", "content": "You are a helpful assistant."}]
#   2. turns = [
#        "Tell me one fact about Python programming.",
#        "Give me another fact.",
#        "And a third fact.",
#        "Summarise all three facts in one sentence.",
#      ]
#   3. For i, turn in enumerate(turns, 1):
#        messages.append({"role": "user", "content": turn})
#        response = ... (call API or mock)
#        messages.append({"role": "assistant", "content": ...})
#        cost = estimate_cost(response.usage, MODEL)
#        print(f"Turn {i}: prompt={response.usage.prompt_tokens} "
#              f"completion={response.usage.completion_tokens} "
#              f"cost=${cost:.6f}")
# --- live-code below ---




> **Exercise 1 — 3-Turn Domain Conversation**
>
> Pick a professional domain (healthcare, logistics, retail, education, agriculture…).
> 1. Write a `system` message giving the model an expert persona in that domain.
> 2. Build a 3-turn conversation where each user message builds on the previous answer.
> 3. After each turn print the reply and the cumulative token count.
>
> Save your `system` message in `MY_SYSTEM` — you'll carry it through all six sessions.

In [23]:
# your turn:
MY_SYSTEM = "You are a script writer famous for creating engaging movie scripts"  # TODO: write your domain persona here


# initializing messages array with system prompt
messages = [
    {'role':'system',"content":MY_SYSTEM}
]

# function to conduct multiturn conversation
def multiturn_conversation(messages,user_question):

    'function to enable multi turn conversation by appending user query and response from model'

    messages.append({'role':'user',"content":user_question})

    response = client.chat.completions.create(
        messages = messages,
        model=MODEL
    )

    messages.append({'role':'assistant',"content":response.choices[0].message.content})


    return response


pretty_print(multiturn_conversation(messages,'Create a script similar to toy story engaging the kids'))

print(messages)






────────────────────────────────────────────────────────────
**Title: _Toy Town Adventures_**

**Genre:** Animated Family Adventure  
**Style:** Warm, funny, heartful, toy-filled adventure  
**Note:** This is an original story with a playful, kid-friendly vibe inspired by the magic of toys coming to life.

---

## **CHARACTERS**
- **TILLY** – A brave little red robot with a big imagination
- **BOUNCY** – A hyper bouncing ball who never stops talking
- **MIMI** – A fancy plush cat who acts dramatic but cares deeply
- **CAPTAIN KAZ** – A wooden toy astronaut, determined and noble
- **ZIGGY** – A wind-up raccoon with sneaky tricks and a good heart
- **LUNA** – A kind 7-year-old girl who loves all her toys
- **MRS. MAPLE** – Luna’s grandmother, gentle and wise

---

# **TOY TOWN ADVENTURES**

## **FADE IN:**

### **INT. LUNA’S BEDROOM – NIGHT**

Moonlight shines through the window. The room is quiet.

A shelf full of toys sits perfectly still.

Then—

A tiny *click!*

TILLY’s eyes light up

In [24]:
pretty_print(multiturn_conversation(messages,'How can I make this interesting to adults as well.'))

────────────────────────────────────────────────────────────
To make it interesting to adults too, add **layers**: keep the child-friendly fun on the surface, but give adults something emotional, clever, or relatable underneath.

Here are the best ways to do that:

## 1. Give the story a deeper emotional theme
Adults connect to:
- **growing up**
- **letting go**
- **nostalgia**
- **family memory**
- **purpose after loss**
- **the fear of being forgotten**

For example, the toy train isn’t just a toy—it could be tied to Luna’s grandpa, and the adults understand it represents memory, grief, and family connection.

## 2. Add jokes adults will catch
Use humor that works on two levels:
- Kids enjoy the silly action
- Adults enjoy the irony, sarcasm, or realistic parenting moments

Examples:
- A toy says something dramatic like, “I have not slept in 14 years emotionally.”
- Mrs. Maple makes a dry comment like, “That’s what I call a maintenance issue.”
- The toys argue like a dysfunctional wo

---
## 3. Key Chat Completions Parameters

Beyond `model` and `messages`, three parameters you will reach for constantly:

| Parameter | Type | Default | What it does |
|-----------|------|---------|-------------|
| `temperature` | float 0–2 | 1.0 | Controls randomness. `0` = nearly deterministic; `1.2+` = highly creative. For factual/structured tasks keep ≤ 0.3. |
| `max_tokens` | int | None (model max) | Hard cap on completion length. When hit, `finish_reason="length"`. Use to control cost. |
| `stop` | str or list[str] | None | The model stops generating as soon as it produces one of these strings. Useful for structured outputs. |

Other parameters worth knowing (rarely need tuning):
- `top_p` — nucleus sampling; usually leave at 1.0 and adjust `temperature` instead
- `frequency_penalty` / `presence_penalty` — reduce word repetition (range 0–2)
- `n` — generate N completions in one call; multiplies cost by N

In [25]:
# ── 3a. Temperature: creativity vs consistency ───────────────────────────────
# GOAL: Feel the difference between low and high temperature first-hand
# STEPS:
#   1. prompt = "Write a two-word tagline for an Indian fintech startup."
#   2. For temperature in [0.0, 0.7, 1.5]:
#        response = client.chat.completions.create(
#            model=MODEL,
#            messages=[{"role": "user", "content": prompt}],
#            temperature=temperature,
#        )
#        print(f"temp={temperature}: {response.choices[0].message.content.strip()}")
#   3. Call temperature=0.0 twice in a row — observe near-identical output
#   4. Call temperature=1.5 twice — observe variation
# --- live-code below ---


prompt = "Write a two-word tagline for an Indian fintech startup."

for temperature in [0.0, 0.7, 1.5]:
    
       response = client.chat.completions.create(
           model=MODEL,
           messages=[{"role": "user", "content": prompt}],
           temperature=temperature,
       )
       print(f"temp={temperature}: {response.choices[0].message.content.strip()}")


temp=0.0: Smart Finance
temp=0.7: Trusted Finance
temp=1.5: Smart Money


In [ ]:
# ── 3b. max_tokens caps length; stop sequences end generation early ──────────
# GOAL: See finish_reason="length" and understand the stop parameter
#
# Part 1 — max_tokens:
#   response = client.chat.completions.create(
#       model=MODEL,
#       messages=[{"role": "user", "content": "Count from 1 to 20, one number per line."}],
#       max_tokens=12,
#   )
#   print(response.choices[0].message.content)
#   print("finish_reason:", response.choices[0].finish_reason)  # expect "length"
#
# Part 2 — stop sequence:
#   response = client.chat.completions.create(
#       model=MODEL,
#       messages=[{"role": "user", "content": "List 5 Indian cities, one per line."}],
#       stop=["3."],   # stops as soon as it writes "3."
#   )
#   print(response.choices[0].message.content)
#   print("finish_reason:", response.choices[0].finish_reason)  # expect "stop"
# --- live-code below ---


---
## 4. Building a CLI Chatbot

A CLI chatbot is just a `while` loop around three operations:
1. Read user input from the terminal
2. Call the API with the updated `messages` list
3. Append the assistant reply and print it

**Two versions we'll build:**
- **In this notebook:** simulate a conversation with predefined turns (avoids blocking `input()` in Jupyter)
- **As a runnable script:** `apps/chatbot_cli.py` — run from the terminal with `python apps/chatbot_cli.py`

In [ ]:
# ── 4a. chat_turn() — the core building block ───────────────────────────────
# GOAL: One function handles a single exchange and updates history in place
# STEPS:
#   1. def chat_turn(messages: list, user_input: str, **kwargs) -> str:
#        messages.append({"role": "user", "content": user_input})
#        if MOCK:
#            response = mock_chat_response(f"[mock reply to: {user_input}]")
#        else:
#            response = retry(lambda: client.chat.completions.create(
#                                 model=MODEL, messages=messages, **kwargs))
#        reply = response.choices[0].message.content
#        messages.append({"role": "assistant", "content": reply})
#        return reply
#
#   2. Quick test:
#        hist = [{"role": "system", "content": "You are a helpful assistant."}]
#        print(chat_turn(hist, "What is Python?"))
#        print(chat_turn(hist, "Name one data-science library for it."))
# --- live-code below ---


In [ ]:
# ── 4b. Simulate a multi-turn conversation in the notebook ───────────────────
# GOAL: Run a full chatbot conversation programmatically (no blocking input())
# STEPS:
#   1. SYSTEM = "You are a career advisor for IT professionals in India."
#      history = [{"role": "system", "content": SYSTEM}]
#   2. turns = [
#        "I have 3 years of Python experience. What should I learn next for AI?",
#        "How long would it take to become job-ready in that area?",
#        "Recommend one free online resource for it.",
#      ]
#   3. For i, turn in enumerate(turns, 1):
#        reply = chat_turn(history, turn)
#        print(f"User: {turn}")
#        print(f"Bot:  {reply}")
#        print(f"      (history: {len(history)} messages)\n")
# --- live-code below ---


---
## 5. Minimal Streamlit Chat UI

[Streamlit](https://streamlit.io) turns a Python script into a web UI. Every user interaction **re-runs the entire script from top to bottom**, so conversation state must live in `st.session_state`.

### Key components for chat
| Component | Purpose |
|-----------|---------|
| `st.chat_message("user")` | Renders a user message bubble |
| `st.chat_message("assistant")` | Renders a model message bubble |
| `st.chat_input("Type here…")` | Input bar fixed to the bottom; returns `None` when empty |
| `st.session_state["messages"]` | Persists the `messages` list across re-runs |

### Running the app
```
streamlit run apps/chat_app.py
```
(always run from the project root)

The skeleton is in `apps/chat_app.py`. We'll walk through it below and fill in the live-code sections during the session.

In [ ]:
# ── 5a. Inspect the Streamlit app skeleton ──────────────────────────────────
# GOAL: Read the source and map each block to its role in the chat flow
# STEPS:
#   1. Run this to display the app source:
#        from pathlib import Path
#        print(Path("apps/chat_app.py").read_text())
#      (if you're not in the project root, adjust the path)
#
#   2. Identify in the printed code:
#        a) Where session_state is initialised (prevents KeyError on first run)
#        b) Where existing messages are rendered in a loop
#        c) Where st.chat_input() captures the new user prompt
#        d) Where the API is called
#        e) Where the assistant reply is appended to session_state
#
#   3. After the walk-through, open a new terminal and run:
#        streamlit run apps/chat_app.py
# --- live-code below ---


---
## Exercises

> **Exercise 2 — Add a System Prompt Selector**
>
> Extend `chat_turn()` (or the Streamlit app) to let the user pick from 3 preset personas:
> - `"Formal advisor"` — professional, no contractions
> - `"Friendly tutor"` — simple language, encouraging
> - `"Devil's advocate"` — challenges every assumption
>
> When the persona changes, **reset the messages list** so the model starts fresh.  
> Hint: in Streamlit use `st.selectbox`; in the CLI, show a numbered menu before the loop.

In [ ]:
# your turn:


> **Exercise 3 — Running Cost Tracker**
>
> Modify your conversation loop (CLI or Streamlit) to:
> 1. Accumulate `total_cost` across all turns using `estimate_cost()`
> 2. Print / display `f"Session cost so far: ${total_cost:.5f}"` after each turn
> 3. Print a `"⚠ Cost exceeded $0.01"` warning if the total crosses that threshold

In [ ]:
# your turn:


---
## Common Pitfalls

| Mistake | Symptom | Fix |
|---------|---------|-----|
| Forgetting to append the assistant reply | Model loses context after Turn 1 | Always `messages.append({"role": "assistant", "content": reply})` after every call |
| Sending only the latest message, not the full list | Model behaves like a fresh single-turn chat | Pass the entire `messages` list to `create()` |
| Very long `system` prompt | High cost multiplied by every turn | Keep `system` tight; move reference material to RAG (Session 5) |
| `finish_reason="length"` on every reply | Answers are always truncated | Increase `max_tokens` or shorten history |
| Streamlit history disappears on refresh | Used a module-level variable instead of `session_state` | Store `messages` in `st.session_state`, not a plain Python variable |
| `temperature > 2.0` | `BadRequestError` from the API | Clamp to 0.0–2.0 |

---
## Recap

- The Chat Completions API is **stateless** — conversation memory is your `messages` list, re-sent in full on every call.
- Three roles: `system` (persona, sent once), `user` (human input), `assistant` (model replies you append).
- `temperature` controls creativity; `max_tokens` caps output cost; `stop` gives precise control over where generation ends.
- A CLI chatbot is just a `while` loop around `chat_turn()` — the helper handles append logic for you.
- In Streamlit every interaction re-runs the script, so store `messages` in `st.session_state` to keep history alive.

---
## Homework / Capstone Increment

Before Session 3 (Sat 27 Jun):

1. **Run the CLI chatbot:** `python apps/chatbot_cli.py` from the project root. Have a 5-turn conversation and note the token count after each turn.

2. **Run the Streamlit app:** `streamlit run apps/chat_app.py`. Verify that the conversation history persists across messages in the browser.

3. **Capstone thread:** Plug your `MY_SYSTEM` persona (from Session 1) into `chat_app.py` as the default system message.

4. **Optional — history trimming:** Modify `chat_turn()` to keep at most the system message + the last 6 messages (3 turns). Does the chatbot still feel coherent? Why or why not?

5. **Optional — temperature tuning:** Find the `temperature` value that produces the best balance of creativity and accuracy for your chosen domain. Note it down — we'll use it in Session 3 when we cover structured outputs.